# Paper-wide public-data acquisition

This notebook is the **only data-acquisition notebook** for the paper.
It downloads the public, analysis-ready source files used by Figures
1–7 and records their provenance and observed SHA-256 hashes.

It does **not** contain manuscript coefficients, correlations, P values,
confidence intervals, or plotting tables. Figure-specific analysis
notebooks must derive those outputs from the files acquired here.

Public deposited matrices and study workbooks are treated as source
data. Reprocessing FASTQ/BAM files is outside the reproducibility scope.

## Source map

| Cohort/source | Public source object | Used in figures |
|---|---|---:|
| TCGA-KIRC | UCSC Xena HiSeqV2 matrix; GDC clinical API; PanCanAtlas ABSOLUTE purity | 1–3 |
| CPTAC ccRCC | LinkedOmics RNA/protein matrices; published clinical supplement; CPTAC package phosphoproteomics | 1, 3, 5 |
| E-MTAB-1980 | BioStudies expression/platform files; Sato clinical supplement | 1, 3 |
| TRACERx Renal | Author Figshare code/data bundle | 1, 3, 4 |
| GSE159115 | GEO single-cell HDF5 archive and ccRCC annotation | 5 |
| GSE210041 | GEO spatial HDF5/spatial archive | 5 |
| JAVELIN Renal 101 | Original published supplementary tables workbook | 5–6 |
| CheckMate 025 | Original published Supplementary Table 7 workbook | 6 |
| GSE235046 | GEO source-experiment count table | 7 |

In [10]:
from pathlib import Path
import concurrent.futures as cf
import csv
import hashlib
import importlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import zipfile

import pandas as pd
import requests


PROJECT_ROOT = Path("/scratch/ar58064/mitoxy")
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROVENANCE_ROOT = PROJECT_ROOT / "data" / "provenance"
RAW_ROOT.mkdir(parents=True, exist_ok=True)
PROVENANCE_ROOT.mkdir(parents=True, exist_ok=True)

# Set any item to False only when intentionally reusing a previously
# verified local copy. The final audit still requires every source.
DOWNLOAD = {
    "tcga": True,
    "cptac": True,
    "emtab": True,
    "tracerx": True,
    "gse159115": True,
    "gse210041": True,
    "javelin": True,
    "checkmate025": True,
    "gse235046": True,
}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "mitoxy-public-reproduction/1.0"})
MANIFEST_ROWS = []


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def record_source(dataset, role, path, url, accession=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    row = {
        "dataset": dataset,
        "accession": accession,
        "role": role,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "source_url": url,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    MANIFEST_ROWS.append(row)
    return row


def download_file(url, destination, expected_size=None,
                  expected_sha256=None, label=None,
                  fallback_urls=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        size_ok = expected_size is None or destination.stat().st_size == expected_size
        hash_ok = expected_sha256 is None or sha256_file(destination) == expected_sha256
        if size_ok and hash_ok:
            print("Verified existing:", destination)
            return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    candidates = [url, *(fallback_urls or [])]
    errors = []
    for candidate in candidates:
        temporary.unlink(missing_ok=True)
        print("Downloading:", label or destination.name)
        print("Source:", candidate)
        try:
            if candidate.startswith("ftp://"):
                completed = subprocess.run(
                    [
                        "curl", "--fail", "--location",
                        "--retry", "4", "--retry-delay", "3",
                        "--connect-timeout", "60", "--max-time", "7200",
                        "--user-agent", "mitoxy-public-reproduction/1.0",
                        "--output", str(temporary), candidate,
                    ],
                    check=False,
                    capture_output=True,
                    text=True,
                )
                if completed.returncode != 0:
                    raise RuntimeError(completed.stderr.strip())
            else:
                with SESSION.get(
                    candidate,
                    stream=True,
                    timeout=(60, 1200),
                ) as response:
                    response.raise_for_status()
                    with temporary.open("wb") as handle:
                        for chunk in response.iter_content(1024 * 1024):
                            if chunk:
                                handle.write(chunk)

            observed_size = temporary.stat().st_size
            observed_sha = sha256_file(temporary)
            if expected_size is not None and observed_size != expected_size:
                raise RuntimeError(
                    f"size {observed_size} != {expected_size}"
                )
            if expected_sha256 is not None and observed_sha != expected_sha256:
                raise RuntimeError(
                    f"SHA-256 {observed_sha} != {expected_sha256}"
                )
            temporary.replace(destination)
            return destination
        except Exception as error:
            temporary.unlink(missing_ok=True)
            errors.append(f"{candidate}: {error}")

    raise RuntimeError(
        f"All download routes failed for {destination.name}:\n"
        + "\n".join(errors)
    )


def write_json(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    return destination


def safe_extract_tar(archive, destination):
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as handle:
        root = destination.resolve()
        for member in handle.getmembers():
            target = (destination / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f"Unsafe tar member: {member.name}")
        handle.extractall(destination)


def download_pmc_attachment(pmcid, member_name, destination,
                            expected_size, expected_sha256):
    destination = Path(destination)
    if (destination.exists()
            and destination.stat().st_size == expected_size
            and sha256_file(destination) == expected_sha256):
        print("Verified existing:", destination)
        return destination

    instance = str(pmcid).upper().removeprefix("PMC")
    url = (
        "https://pmc.ncbi.nlm.nih.gov/articles/instance/"
        f"{instance}/bin/{member_name}"
    )
    response = SESSION.get(url, timeout=300)
    response.raise_for_status()
    payload = response.content

    if len(payload) != expected_size:
        page = response.text
        challenge = re.search(
            r'const POW_CHALLENGE = "([^"]+)"', page
        )
        difficulty = re.search(
            r'const POW_DIFFICULTY = "(\d+)"', page
        )
        cookie_name = re.search(
            r'const POW_COOKIE_NAME = "([^"]+)"', page
        )
        if not all([challenge, difficulty, cookie_name]):
            raise RuntimeError(
                f"Unexpected PMC response for {member_name}"
            )
        challenge = challenge.group(1)
        difficulty = int(difficulty.group(1))
        cookie_name = cookie_name.group(1)
        nonce, prefix = 0, "0" * difficulty
        while not hashlib.sha256(
            f"{challenge}{nonce}".encode()
        ).hexdigest().startswith(prefix):
            nonce += 1
        headers = {
            "Referer": url,
            "Cookie": f"{cookie_name}={challenge}%2C{nonce}",
        }
        response = SESSION.get(url, headers=headers, timeout=1200)
        response.raise_for_status()
        payload = response.content

    observed = hashlib.sha256(payload).hexdigest()
    if len(payload) != expected_size or observed != expected_sha256:
        raise RuntimeError(
            f"PMC integrity failure for {member_name}: "
            f"size={len(payload)}, sha256={observed}"
        )
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.write_bytes(payload)
    temporary.replace(destination)
    return destination

## 1. TCGA-KIRC

In [11]:
tcga_dir = RAW_ROOT / "tcga_kirc_public"
tcga_dir.mkdir(parents=True, exist_ok=True)

tcga_expression_url = (
    "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
    "TCGA.KIRC.sampleMap%2FHiSeqV2.gz"
)
tcga_expression = tcga_dir / "TCGA.KIRC.sampleMap_HiSeqV2.gz"
purity_url = "https://api.gdc.cancer.gov/data/4f277128-f793-4354-a13d-30cc7fe9f6b5"
purity_path = tcga_dir / "TCGA_mastercalls.abs_tables_JSedit.fixed.txt"

if DOWNLOAD["tcga"]:
    download_file(
        tcga_expression_url, tcga_expression,
        # 30.85 MiB when displayed with a 1024-based divisor.
        # The exact object length is 32,349,770 bytes.
        expected_size=32_349_770,
        expected_sha256="17717853fc21d4f0e40239827a01e73486a61ab47def732b29999f0667a3b6da",
    )
    download_file(
        purity_url, purity_path,
        expected_size=901_812,
        expected_sha256="f430a975433d82e0098d7405619d4f12a0c765fcd97e7d63cc9b1de7f2d763cd",
    )

    cases_url = "https://api.gdc.cancer.gov/cases"
    case_payload = {
        "filters": {"op": "in", "content": {
            "field": "project.project_id", "value": ["TCGA-KIRC"]}},
        "format": "JSON", "size": 2000,
        "expand": "diagnoses,demographic",
        "fields": "submitter_id,case_id,primary_site,disease_type",
    }
    response = SESSION.post(cases_url, json=case_payload, timeout=300)
    response.raise_for_status()
    clinical_payload_path = write_json(
        case_payload, tcga_dir / "gdc_clinical_query_payload.json"
    )
    clinical_response_path = write_json(
        response.json(), tcga_dir / "gdc_clinical_query_response.json"
    )

record_source("TCGA-KIRC", "tumor RNA expression", tcga_expression,
              tcga_expression_url, "TCGA-KIRC")
record_source("TCGA-KIRC", "ABSOLUTE purity", purity_path,
              purity_url, "4f277128-f793-4354-a13d-30cc7fe9f6b5")
record_source("TCGA-KIRC", "clinical API response",
              tcga_dir / "gdc_clinical_query_response.json",
              "https://api.gdc.cancer.gov/cases", "TCGA-KIRC")

Verified existing: /scratch/ar58064/mitoxy/data/raw/tcga_kirc_public/TCGA.KIRC.sampleMap_HiSeqV2.gz
Verified existing: /scratch/ar58064/mitoxy/data/raw/tcga_kirc_public/TCGA_mastercalls.abs_tables_JSedit.fixed.txt


{'dataset': 'TCGA-KIRC',
 'accession': 'TCGA-KIRC',
 'role': 'clinical API response',
 'relative_path': 'data/raw/tcga_kirc_public/gdc_clinical_query_response.json',
 'source_url': 'https://api.gdc.cancer.gov/cases',
 'size_bytes': 1437633,
 'sha256': 'a7b7a4ab756da3cd4644487ab9e9627a32c0c1fa577b4ccd2464dcfbeb5bf65c'}

## 2. CPTAC ccRCC

In [12]:
cptac_dir = RAW_ROOT / "cptac_ccrcc_public"
cptac_dir.mkdir(parents=True, exist_ok=True)
cptac_rna_url = (
    "https://linkedomics.org/data_download/CPTAC-CCRCC/"
    "HS_CPTAC_CCRCC_RNAseq_fpkm_log2_Tumor.cct"
)
cptac_protein_url = (
    "https://linkedomics.org/data_download/CPTAC-CCRCC/"
    "HS_CPTAC_CCRCC_proteome_Tumor.cct"
)
cptac_rna = cptac_dir / "HS_CPTAC_CCRCC_RNAseq_fpkm_log2_Tumor.txt"
cptac_protein = cptac_dir / "HS_CPTAC_CCRCC_proteome_Tumor.txt"
cptac_clinical = cptac_dir / "NIHMS1597713-supplement-Table_S1.xlsx"

gdc_files_url = "https://api.gdc.cancer.gov/files"
gdc_data_url = "https://api.gdc.cancer.gov/data"
gdc_index = cptac_dir / "cptac3_kidney_masked_maf_file_index.json"
gdc_bundle = cptac_dir / "cptac3_kidney_masked_mafs.tar.gz"


def gdc_bundle_file_ids(path):
    path = Path(path)
    if not path.exists():
        return set()
    try:
        with tarfile.open(path, mode="r:gz") as archive:
            return {
                member.name.split("/", 1)[0]
                for member in archive.getmembers()
                if member.isfile() and member.name.endswith(".maf.gz")
            }
    except (tarfile.TarError, OSError):
        return set()


def download_gdc_bundle(file_ids, destination, max_attempts=4):
    destination = Path(destination)
    expected_ids = set(file_ids)
    if gdc_bundle_file_ids(destination) == expected_ids:
        print("Verified existing:", destination)
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    errors = []
    for attempt in range(1, max_attempts + 1):
        temporary.unlink(missing_ok=True)
        try:
            print(
                "Downloading open GDC CPTAC-3 masked MAF bundle "
                f"(attempt {attempt}/{max_attempts})"
            )
            with SESSION.post(
                gdc_data_url,
                json={"ids": sorted(expected_ids)},
                stream=True,
                timeout=(60, 1800),
            ) as response:
                response.raise_for_status()
                with temporary.open("wb") as handle:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            handle.write(chunk)

            observed_ids = gdc_bundle_file_ids(temporary)
            if observed_ids != expected_ids:
                raise RuntimeError(
                    "GDC bundle/file-index disagreement: "
                    f"expected {len(expected_ids)} files, "
                    f"found {len(observed_ids)}"
                )
            temporary.replace(destination)
            return destination
        except Exception as error:
            temporary.unlink(missing_ok=True)
            errors.append(f"attempt {attempt}: {error}")
            if attempt < max_attempts:
                time.sleep(min(2 ** attempt, 15))

    raise RuntimeError(
        "Unable to download the open CPTAC-3 masked-MAF bundle.\n"
        + "\n".join(errors)
    )


if DOWNLOAD["cptac"]:
    download_file(
        cptac_rna_url, cptac_rna, 23_037_690,
        "f7cd6fe435c1be7cfe902780432e637d277ea7bb6e63d1805647a29004f3b1c9",
    )
    download_file(
        cptac_protein_url, cptac_protein, 19_368_136,
        "16e75796570a56d854094d119140587814026eb6356240596fa3667da513e9c2",
    )
    download_pmc_attachment(
        "PMC7331093", "NIHMS1597713-supplement-Table_S1.xlsx",
        cptac_clinical, 120_867,
        "ecd4647846a2f401319d7090a5d689a84a10bcd375ceba25c1355aaa47144c77",
    )

    # The source CPTAC package is retained only for the public UMich
    # phosphoproteomic table used in Figure 3. The package's WashU ccRCC
    # mutation filename is not present in its current Zenodo release, so
    # mutation calls are acquired directly from the open GDC CPTAC-3 files.
    cptac_version = "1.5.14"
    try:
        installed = importlib.metadata.version("cptac")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != cptac_version:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            f"cptac=={cptac_version}",
        ])
    import cptac
    cc = cptac.Ccrcc()
    phospho = cc.get_phosphoproteomics(
        source="umich", tissue_type="tumor"
    )
    phospho_export = cptac_dir / "phosphoproteomics_umich.pkl.gz"
    phospho.to_pickle(phospho_export, compression="gzip")

    gdc_filters = {
        "op": "and",
        "content": [
            {
                "op": "=",
                "content": {
                    "field": "cases.project.project_id",
                    "value": ["CPTAC-3"],
                },
            },
            {
                "op": "=",
                "content": {
                    "field": "cases.primary_site",
                    "value": ["Kidney"],
                },
            },
            {
                "op": "=",
                "content": {
                    "field": "data_type",
                    "value": ["Masked Somatic Mutation"],
                },
            },
        ],
    }
    gdc_fields = [
        "file_id", "file_name", "file_size", "md5sum", "data_type",
        "data_format", "access", "experimental_strategy",
        "analysis.workflow_type", "cases.case_id", "cases.submitter_id",
        "cases.samples.submitter_id",
    ]
    response = SESSION.get(
        gdc_files_url,
        params={
            "filters": json.dumps(gdc_filters, separators=(",", ":")),
            "fields": ",".join(gdc_fields),
            "format": "JSON",
            "size": "1000",
            "sort": "file_id:asc",
        },
        timeout=(60, 1200),
    )
    response.raise_for_status()
    gdc_payload = response.json()
    gdc_hits = gdc_payload.get("data", {}).get("hits", [])
    if not gdc_hits:
        raise RuntimeError("The GDC query returned no CPTAC-3 kidney masked MAFs")
    if any(hit.get("access") != "open" for hit in gdc_hits):
        raise RuntimeError("The GDC masked-MAF query returned a controlled file")
    gdc_file_ids = sorted({hit["file_id"] for hit in gdc_hits})
    if len(gdc_file_ids) != len(gdc_hits):
        raise RuntimeError("The GDC file index contains duplicate file IDs")

    write_json(gdc_payload, gdc_index)
    download_gdc_bundle(gdc_file_ids, gdc_bundle)

    package_cache = cptac_dir / "package_source_cache"
    source = Path(cptac.CPTAC_BASE_DIR) / "data" / "umich-ccrcc"
    if source.exists():
        shutil.copytree(
            source, package_cache / "umich-ccrcc", dirs_exist_ok=True
        )

# Keep CPTAC provenance idempotent if this cell is rerun.
MANIFEST_ROWS[:] = [
    row for row in MANIFEST_ROWS if row["dataset"] != "CPTAC ccRCC"
]

record_source("CPTAC ccRCC", "tumor RNA expression", cptac_rna,
              cptac_rna_url, "CPTAC-CCRCC")
record_source("CPTAC ccRCC", "tumor protein abundance", cptac_protein,
              cptac_protein_url, "CPTAC-CCRCC")
record_source("CPTAC ccRCC", "clinical supplement", cptac_clinical,
              "https://pmc.ncbi.nlm.nih.gov/articles/PMC7331093/",
              "PMC7331093")
record_source("CPTAC ccRCC", "phosphoproteomics package export",
              cptac_dir / "phosphoproteomics_umich.pkl.gz",
              "https://github.com/PayneLab/cptac", "cptac==1.5.14")
record_source("CPTAC ccRCC", "GDC masked somatic mutation file index",
              gdc_index, gdc_files_url, "CPTAC-3")
record_source("CPTAC ccRCC", "GDC masked somatic mutation bundle",
              gdc_bundle, gdc_data_url, "CPTAC-3")


Verified existing: /scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_public/HS_CPTAC_CCRCC_RNAseq_fpkm_log2_Tumor.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_public/HS_CPTAC_CCRCC_proteome_Tumor.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_public/NIHMS1597713-supplement-Table_S1.xlsx


/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/cptac/cancers/umich/umichccrcc.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[['Database_ID','Transcript_ID',"Gene_ID","Havana_gene","Havana_transcript","Transcript","Name","Site"]] = df.Index.str.split("\\|",expand=True)
/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/cptac/cancers/umich/umichccrcc.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[['num1','start',"end","detected_phos","localized_phos","Site"]

Verified existing: /scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_public/cptac3_kidney_masked_mafs.tar.gz


{'dataset': 'CPTAC ccRCC',
 'accession': 'CPTAC-3',
 'role': 'GDC masked somatic mutation bundle',
 'relative_path': 'data/raw/cptac_ccrcc_public/cptac3_kidney_masked_mafs.tar.gz',
 'source_url': 'https://api.gdc.cancer.gov/data',
 'size_bytes': 9867765,
 'sha256': '2b30150256afd8d9eb0e0008dd4e7e42faed75f7d45e18597c5034e7628ee3a4'}

## 3. E-MTAB-1980

In [13]:
emtab_dir = RAW_ROOT / "e_mtab_1980_public"
emtab_dir.mkdir(parents=True, exist_ok=True)
emtab_base = "https://www.ebi.ac.uk/biostudies/files/E-MTAB-1980"
emtab_ftp_base = (
    "ftp://ftp.ebi.ac.uk/biostudies/fire/"
    "E-MTAB-/980/E-MTAB-1980/Files"
)
array_ftp_base = (
    "ftp://ftp.ebi.ac.uk/biostudies/fire/"
    "A-MEXP-/183/A-MEXP-2183/Files"
)
emtab_sources = {
    "ccRCC_exp_log_quantile_normalized.txt":
        f"{emtab_base}/ccRCC_exp_log_quantile_normalized.txt",
    "E-MTAB-1980.sdrf.txt": f"{emtab_base}/E-MTAB-1980.sdrf.txt",
    "E-MTAB-1980.idf.txt": f"{emtab_base}/E-MTAB-1980.idf.txt",
    "A-MEXP-2183.adf.txt":
        "https://www.ebi.ac.uk/biostudies/files/A-MEXP-2183/A-MEXP-2183.adf.txt",
    "Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx": (
        "https://media.springernature.com/original/springer-static/esm/"
        "art%3A10.1038%2Fng.2699/MediaObjects/"
        "41588_2013_BFng2699_MOESM35_ESM.xlsx"
    ),
}
emtab_expected_sizes = {
    "ccRCC_exp_log_quantile_normalized.txt": 53_072_899,
    "E-MTAB-1980.sdrf.txt": 60_861,
    "E-MTAB-1980.idf.txt": 2_949,
    "A-MEXP-2183.adf.txt": 7_261_766,
}
if DOWNLOAD["emtab"]:
    for filename, url in emtab_sources.items():
        fallback_urls = []
        if filename == "A-MEXP-2183.adf.txt":
            fallback_urls.append(f"{array_ftp_base}/{filename}")
        elif filename in emtab_expected_sizes:
            fallback_urls.append(f"{emtab_ftp_base}/{filename}")
        download_file(
            url,
            emtab_dir / filename,
            expected_size=emtab_expected_sizes.get(filename),
            fallback_urls=fallback_urls,
        )
    api_url = "https://www.ebi.ac.uk/biostudies/api/v1/studies/E-MTAB-1980"
    response = SESSION.get(api_url, timeout=300)
    response.raise_for_status()
    write_json(response.json(), emtab_dir / "E-MTAB-1980_biostudies_record.json")

for filename, url in emtab_sources.items():
    record_source("E-MTAB-1980", filename, emtab_dir / filename,
                  url, "E-MTAB-1980")

Verified existing: /scratch/ar58064/mitoxy/data/raw/e_mtab_1980_public/ccRCC_exp_log_quantile_normalized.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/e_mtab_1980_public/E-MTAB-1980.sdrf.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/e_mtab_1980_public/E-MTAB-1980.idf.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/e_mtab_1980_public/A-MEXP-2183.adf.txt
Verified existing: /scratch/ar58064/mitoxy/data/raw/e_mtab_1980_public/Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx


## 4. TRACERx Renal

In [14]:
tracerx_dir = RAW_ROOT / "tracerx_renal_public"
tracerx_dir.mkdir(parents=True, exist_ok=True)
figshare_id = 28060796
api_url = f"https://api.figshare.com/v2/articles/{figshare_id}"
archive_path = tracerx_dir / "tx100_rna_2024.zip"

if DOWNLOAD["tracerx"]:
    article = SESSION.get(api_url, timeout=300)
    article.raise_for_status()
    metadata = article.json()
    write_json(metadata, tracerx_dir / "figshare_article_record.json")
    matches = [x for x in metadata["files"]
               if x["name"] == "tx100_rna_2024.zip"]
    if len(matches) != 1:
        raise RuntimeError(f"TRACERx archive match failure: {matches}")
    source_file = matches[0]
    download_file(source_file["download_url"], archive_path,
                  expected_size=source_file.get("size"))

    selected = tracerx_dir / "selected_public_files"
    wanted_suffixes = {
        "data/processed/tumour_tpm.rds",
        "data/meta/tx_annotation.tsv",
        "data/meta/TRACERx_s1_1_clinical.txt",
        "analysis/outputs/sup_tables/Supplementary_Table1.xlsx",
        "README.md",
    }
    with zipfile.ZipFile(archive_path) as archive:
        for name in archive.namelist():
            normalized = name.replace("\\", "/")
            match = next((suffix for suffix in wanted_suffixes
                          if normalized.endswith(suffix)), None)
            if match is None:
                continue
            target = selected / match
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(name) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)

record_source("TRACERx Renal", "author data/code archive", archive_path,
              api_url, f"Figshare {figshare_id}")
for relative, role in [
    ("data/processed/tumour_tpm.rds", "regional tumor TPM"),
    ("data/meta/tx_annotation.tsv", "regional annotation"),
    ("data/meta/TRACERx_s1_1_clinical.txt", "clinical annotation"),
    ("analysis/outputs/sup_tables/Supplementary_Table1.xlsx",
     "evolutionary/genomic supplement"),
]:
    record_source("TRACERx Renal", role,
                  tracerx_dir / "selected_public_files" / relative,
                  api_url, f"Figshare {figshare_id}")

Verified existing: /scratch/ar58064/mitoxy/data/raw/tracerx_renal_public/tx100_rna_2024.zip


## 5. GEO single-cell and spatial datasets

In [15]:
geo_sources = {
    "gse159115": {
        "accession": "GSE159115",
        "directory": RAW_ROOT / "gse159115_public",
        "files": {
            "GSE159115_RAW.tar": (
                "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE159nnn/"
                "GSE159115/suppl/GSE159115_RAW.tar", 120_535_040),
            "GSE159115_ccRCC_anno.csv.gz": (
                "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE159nnn/"
                "GSE159115/suppl/GSE159115_ccRCC_anno.csv.gz", 927_306),
        },
    },
    "gse210041": {
        "accession": "GSE210041",
        "directory": RAW_ROOT / "gse210041_public",
        "files": {
            "GSE210041_RAW.tar": (
                "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nnn/"
                "GSE210041/suppl/GSE210041_RAW.tar", 122_531_840),
        },
    },
}

for key, entry in geo_sources.items():
    directory = entry["directory"]
    directory.mkdir(parents=True, exist_ok=True)
    if DOWNLOAD[key]:
        for filename, (url, size) in entry["files"].items():
            download_file(url, directory / filename, expected_size=size)
        safe_extract_tar(directory / next(
            name for name in entry["files"] if name.endswith(".tar")
        ), directory / "extracted")
    for filename, (url, _) in entry["files"].items():
        record_source(entry["accession"], filename,
                      directory / filename, url, entry["accession"])

Verified existing: /scratch/ar58064/mitoxy/data/raw/gse159115_public/GSE159115_RAW.tar
Verified existing: /scratch/ar58064/mitoxy/data/raw/gse159115_public/GSE159115_ccRCC_anno.csv.gz
Verified existing: /scratch/ar58064/mitoxy/data/raw/gse210041_public/GSE210041_RAW.tar


## 6. JAVELIN Renal 101 and CheckMate 025

In [16]:
javelin_dir = RAW_ROOT / "javelin_renal_101_public"
checkmate_dir = RAW_ROOT / "checkmate_025_public"
javelin_path = javelin_dir / "NIHMS1737783-supplement-Supplementary_Tables.xlsb"
checkmate_path = checkmate_dir / "NIHMS1823968-supplement-7.xlsx"

if DOWNLOAD["javelin"]:
    download_pmc_attachment(
        "PMC8493486",
        "NIHMS1737783-supplement-Supplementary_Tables.xlsb",
        javelin_path,
        93_320_749,
        "7bcfb502eceaff4e655e3f313d6e37fe6fe73c01b3bb846ad1bb8c8d6bd2ba97",
    )
if DOWNLOAD["checkmate025"]:
    download_pmc_attachment(
        "PMC9481706", "NIHMS1823968-supplement-7.xlsx",
        checkmate_path,
        67_344_494,
        "061cb25f083c75f1e544931d9c9b0be8a91ebcac46c86c78274ded2d97ce174e",
    )

record_source("JAVELIN Renal 101", "clinical and molecular tables",
              javelin_path,
              "https://pmc.ncbi.nlm.nih.gov/articles/PMC8493486/",
              "PMC8493486")
record_source("CheckMate 025", "clinical and molecular tables",
              checkmate_path,
              "https://pmc.ncbi.nlm.nih.gov/articles/PMC9481706/",
              "PMC9481706")

Verified existing: /scratch/ar58064/mitoxy/data/raw/javelin_renal_101_public/NIHMS1737783-supplement-Supplementary_Tables.xlsb
Verified existing: /scratch/ar58064/mitoxy/data/raw/checkmate_025_public/NIHMS1823968-supplement-7.xlsx


{'dataset': 'CheckMate 025',
 'accession': 'PMC9481706',
 'role': 'clinical and molecular tables',
 'relative_path': 'data/raw/checkmate_025_public/NIHMS1823968-supplement-7.xlsx',
 'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9481706/',
 'size_bytes': 67344494,
 'sha256': '061cb25f083c75f1e544931d9c9b0be8a91ebcac46c86c78274ded2d97ce174e'}

## 7. GSE235046 source experiment

In [17]:
gse235046_dir = RAW_ROOT / "gse235046_public"
gse235046_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE235nnn/"
    "GSE235046/suppl/GSE235046_Count_table.txt.gz"
)
gse235046_path = gse235046_dir / "GSE235046_Count_table.txt.gz"
if DOWNLOAD["gse235046"]:
    download_file(
        gse235046_url, gse235046_path,
        expected_size=1_110_939,
        expected_sha256="683ec07b3d81ea8cb6b834b08e071ba9ec2b89f49c61ac303029864b1a110906",
    )
record_source("GSE235046", "BMDM count matrix", gse235046_path,
              gse235046_url, "GSE235046")

Verified existing: /scratch/ar58064/mitoxy/data/raw/gse235046_public/GSE235046_Count_table.txt.gz


{'dataset': 'GSE235046',
 'accession': 'GSE235046',
 'role': 'BMDM count matrix',
 'relative_path': 'data/raw/gse235046_public/GSE235046_Count_table.txt.gz',
 'source_url': 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE235nnn/GSE235046/suppl/GSE235046_Count_table.txt.gz',
 'size_bytes': 1110939,
 'sha256': '683ec07b3d81ea8cb6b834b08e071ba9ec2b89f49c61ac303029864b1a110906'}

## Final provenance manifest and completeness audit

In [18]:
manifest = (
    pd.DataFrame(MANIFEST_ROWS)
    .drop_duplicates(subset=["dataset", "role", "relative_path"])
    .sort_values(["dataset", "role"])
    .reset_index(drop=True)
)
manifest_path = PROVENANCE_ROOT / "paper_public_source_manifest.csv"
manifest.to_csv(manifest_path, index=False)

expected_datasets = {
    "TCGA-KIRC", "CPTAC ccRCC", "E-MTAB-1980", "TRACERx Renal",
    "GSE159115", "GSE210041", "JAVELIN Renal 101",
    "CheckMate 025", "GSE235046",
}
missing = sorted(expected_datasets - set(manifest["dataset"]))
if missing:
    raise AssertionError(f"Missing source datasets: {missing}")
if manifest["sha256"].isna().any() or manifest["size_bytes"].le(0).any():
    raise AssertionError("Manifest contains an empty or unhashed source file")

display(manifest)
print("\nPaper-wide public-data acquisition: PASS")
print("Manifest:", manifest_path)
print("Source objects recorded:", len(manifest))

,dataset,accession,role,relative_path,source_url,size_bytes,sha256
0,CPTAC ccRCC,CPTAC-3,GDC masked somatic mutation bundle,data/raw/cptac_ccrcc_public/cptac3_kidney_mask...,https://api.gdc.cancer.gov/data,9867765,2b30150256afd8d9eb0e0008dd4e7e42faed75f7d45e18...
1,CPTAC ccRCC,CPTAC-3,GDC masked somatic mutation file index,data/raw/cptac_ccrcc_public/cptac3_kidney_mask...,https://api.gdc.cancer.gov/files,328055,d8edb0881fa86c65a7861f05fa2dbe113d9ed69ce1f345...
2,CPTAC ccRCC,PMC7331093,clinical supplement,data/raw/cptac_ccrcc_public/NIHMS1597713-suppl...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7331093/,120867,ecd4647846a2f401319d7090a5d689a84a10bcd375ceba...
3,CPTAC ccRCC,cptac==1.5.14,phosphoproteomics package export,data/raw/cptac_ccrcc_public/phosphoproteomics_...,https://github.com/PayneLab/cptac,19495130,c9511a345357dbe2ea8a0bc952bb6144b219785ebd50cc...
4,CPTAC ccRCC,CPTAC-CCRCC,tumor RNA expression,data/raw/cptac_ccrcc_public/HS_CPTAC_CCRCC_RNA...,https://linkedomics.org/data_download/CPTAC-CC...,23037690,f7cd6fe435c1be7cfe902780432e637d277ea7bb6e63d1...
5,CPTAC ccRCC,CPTAC-CCRCC,tumor protein abundance,data/raw/cptac_ccrcc_public/HS_CPTAC_CCRCC_pro...,https://linkedomics.org/data_download/CPTAC-CC...,19368136,16e75796570a56d854094d119140587814026eb6356240...
6,CheckMate 025,PMC9481706,clinical and molecular tables,data/raw/checkmate_025_public/NIHMS1823968-sup...,https://pmc.ncbi.nlm.nih.gov/articles/PMC9481706/,67344494,061cb25f083c75f1e544931d9c9b0be8a91ebcac46c86c...
7,E-MTAB-1980,E-MTAB-1980,A-MEXP-2183.adf.txt,data/raw/e_mtab_1980_public/A-MEXP-2183.adf.txt,https://www.ebi.ac.uk/biostudies/files/A-MEXP-...,7261766,d1e490431494ea07ab14c1849b61212a862bb2f73b2575...
8,E-MTAB-1980,E-MTAB-1980,E-MTAB-1980.idf.txt,data/raw/e_mtab_1980_public/E-MTAB-1980.idf.txt,https://www.ebi.ac.uk/biostudies/files/E-MTAB-...,2949,7326265769294b5ec3ed58543a0cf1a9606d25a954284e...
9,E-MTAB-1980,E-MTAB-1980,E-MTAB-1980.sdrf.txt,data/raw/e_mtab_1980_public/E-MTAB-1980.sdrf.txt,https://www.ebi.ac.uk/biostudies/files/E-MTAB-...,60861,204d06ca459b70c8a55f5e0cb3220b1967190eefe4a6eb...



Paper-wide public-data acquisition: PASS
Manifest: /scratch/ar58064/mitoxy/data/provenance/paper_public_source_manifest.csv
Source objects recorded: 25
